In [1]:
!pip install -q -U "scikit-learn>=1.8"

In [1]:
import os
os.chdir("/content")
if not os.path.exists("355-mdd-research"):
    !git clone -q https://github.com/jesss77/355-mdd-research.git
os.chdir("/content/355-mdd-research")
!git pull -q
!python src/prepare_data.py
!mkdir -p data/annotation
!wget -q -O data/annotation/GPL570.annot.gz https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPLnnn/GPL570/annot/GPL570.annot.gz
!wget -q -O data/annotation/GPL10558.annot.gz https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL10nnn/GPL10558/annot/GPL10558.annot.gz
!python src/map_genes.py

GSE98793: downloading GEO series matrix...
GSE98793: 192 selected: 128 MDD / 64 controls; excluded 0
  54675 probes; selected values [1.131, 14.390]
GSE32280: downloading GEO series matrix...
GSE32280: 16 selected: 8 MDD / 8 controls; excluded 21
  54675 probes; selected values [1.461, 13.789]
  13 metadata warning(s), see combined manifest
GSE39653: downloading GEO series matrix...
GSE39653: 45 selected: 21 MDD / 24 controls; excluded 8
  43116 probes; selected values [6.445, 14.700]

ONE manifest: metadata/cohort_manifest.csv
Audit summary: metadata/dataset_summary.json
GPL10558: 31,253 unambiguous probe mappings, 20,750 genes
GPL570: 42,904 unambiguous probe mappings, 20,848 genes
GSE98793: 192 samples, 42,904/54,675 probes mapped, 20,848 genes
GSE32280: 16 samples, 42,904/54,675 probes mapped, 20,848 genes
GSE39653: 45 samples, 28,446/43,116 probes mapped, 19,457 genes
Shared genes across all three cohorts: 16,467
  wrote data/processed/GSE98793_shared_genes.tsv.gz
  wrote data/pro

In [2]:
import os, sys, json
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd, sklearn
assert tuple(map(int, sklearn.__version__.split(".")[:2])) >= (1, 8), "upgrade scikit-learn + restart"

import src.external_validation as ev          # Jessy's frozen external-validation code
from src.model_data import load_cohort, EXTERNAL
from src.pipeline_factory import build_model_pipeline
from src.gene_features import build_gene_selector
from src.reference_quantile import ReferenceQuantileNormalizer

# ---- Frozen settings (from config/experiment.yaml, chosen on GSE98793 only) ----
FROZEN = {
    "gene":         dict(pre="passthrough", sel="top50", C=ev.GENE_C,    l1=ev.GENE_L1_RATIO),  # primary (Jessy)
    "gene_qn":      dict(pre="qn",          sel="top50", C=0.1,          l1=0.9),               # sensitivity
    "all_genes":    dict(pre="passthrough", sel="all",   C=10.0,         l1=0.1),               # secondary baseline
}
def build(cfg):
    return build_model_pipeline(
        preprocessor=ReferenceQuantileNormalizer() if cfg["pre"] == "qn" else "passthrough",
        selector=build_gene_selector(k=50) if cfg["sel"] == "top50" else "passthrough",
        C=cfg["C"], l1_ratio=cfg["l1"])

# ---- Fit all models on GSE98793 only (external cohorts not loaded yet) ----
dev = load_cohort("GSE98793")
dev_pw = ev.load_pathway_features("GSE98793", dev.sample_ids)
models = {name: build(cfg).fit(dev.X, dev.y) for name, cfg in FROZEN.items()}
models["pathway"] = build_model_pipeline(selector="passthrough", C=ev.PATHWAY_C,
                                         l1_ratio=ev.PATHWAY_L1_RATIO).fit(dev_pw.X, dev.y)
print("fitted on GSE98793:", list(models))

# ---- Comparisons: (A, B) -> difference = B minus A, same paired bootstrap as Jessy ----
COMPARISONS = [("gene", "gene_qn"),        # does reference QN help the gene model?
               ("gene_qn", "pathway"),     # main question under the QN sensitivity analysis
               ("gene", "all_genes"),      # top-50 vs all shared genes
               ("all_genes", "pathway")]   # pathways vs the full-gene baseline

metric_rows, diff_rows, pred_rows = [], [], []
for cohort in EXTERNAL:
    ext = load_cohort(cohort)
    ev.verify_gene_alignment(dev, ext)
    ext_pw = ev.load_pathway_features(cohort, ext.sample_ids,
                                      expected_feature_names=dev_pw.feature_names)
    probs = {n: m.predict_proba(ext_pw.X if n == "pathway" else ext.X)[:, 1]
             for n, m in models.items()}
    for n, p in probs.items():
        pred_rows += [{"cohort": cohort, "model": n, "GSM": g, "label": int(l), "prob_mdd": float(q)}
                      for g, l, q in zip(ext.sample_ids, ext.y, p)]
    print(f"\n{cohort}: {len(ext.y)} samples")

    cis = {}
    for a, b in COMPARISONS:
        print(f"  bootstrap {b} vs {a}")
        intervals, (lo, hi) = ev.bootstrap_comparison(ext.y, probs[a], probs[b])
        cis.setdefault(a, intervals["gene"]); cis.setdefault(b, intervals["pathway"])
        est = ev.compute_metrics(ext.y, probs[b])[0]["auroc"] - ev.compute_metrics(ext.y, probs[a])[0]["auroc"]
        diff_rows.append({"cohort": cohort, "model_A": a, "model_B": b,
                          "auroc_B_minus_A": est, "ci_low": lo, "ci_high": hi})

    for n in models:
        mets, conf, _ = ev.compute_metrics(ext.y, probs[n])
        for mt, v in mets.items():
            metric_rows.append({"cohort": cohort, "model": n, "metric": mt, "estimate": v,
                                "ci_low": cis[n][mt][0], "ci_high": cis[n][mt][1]})

metrics = pd.DataFrame(metric_rows); diffs = pd.DataFrame(diff_rows)
out = "results/external_secondary"; os.makedirs(out, exist_ok=True)
metrics.to_csv(f"{out}/metrics.csv", index=False)
diffs.to_csv(f"{out}/auroc_differences.csv", index=False)
pd.DataFrame(pred_rows).to_csv(f"{out}/predictions.csv", index=False)
json.dump({"frozen_settings": {**FROZEN, "pathway": dict(C=ev.PATHWAY_C, l1=ev.PATHWAY_L1_RATIO)},
           "bootstrap": dict(replicates=ev.BOOTSTRAP_REPLICATES, random_state=ev.BOOTSTRAP_RANDOM_STATE,
                             method="class_stratified_paired", ci_level=ev.CI_LEVEL),
           "note": "Pre-specified secondary/sensitivity analyses; pathway scores are identical under reference QN"},
          open(f"{out}/settings.json", "w"), indent=2)

# ---- Show results ----
auc = metrics[metrics.metric == "auroc"].copy()
auc["AUROC (95% CI)"] = auc.apply(lambda r: f"{r.estimate:.3f} ({r.ci_low:.2f}-{r.ci_high:.2f})", axis=1)
spec = metrics[metrics.metric == "specificity"].set_index(["cohort", "model"]).estimate.round(2)
tab = auc.set_index(["cohort", "model"])[["AUROC (95% CI)"]].join(spec.rename("specificity"))
print("\n", tab.to_string())
print("\nAUROC differences (B minus A):")
print(diffs.round(3).to_string(index=False))
print("\nCheck: primary gene AUROC should match Jessy's (GSE32280 0.453, GSE39653 0.369)")

fitted on GSE98793: ['gene', 'gene_qn', 'all_genes', 'pathway']

GSE32280: 16 samples
  bootstrap gene_qn vs gene
    bootstrap: 1,000/10,000
    bootstrap: 2,000/10,000
    bootstrap: 3,000/10,000
    bootstrap: 4,000/10,000
    bootstrap: 5,000/10,000
    bootstrap: 6,000/10,000
    bootstrap: 7,000/10,000
    bootstrap: 8,000/10,000
    bootstrap: 9,000/10,000
    bootstrap: 10,000/10,000
  bootstrap pathway vs gene_qn
    bootstrap: 1,000/10,000
    bootstrap: 2,000/10,000
    bootstrap: 3,000/10,000
    bootstrap: 4,000/10,000
    bootstrap: 5,000/10,000
    bootstrap: 6,000/10,000
    bootstrap: 7,000/10,000
    bootstrap: 8,000/10,000
    bootstrap: 9,000/10,000
    bootstrap: 10,000/10,000
  bootstrap all_genes vs gene
    bootstrap: 1,000/10,000
    bootstrap: 2,000/10,000
    bootstrap: 3,000/10,000
    bootstrap: 4,000/10,000
    bootstrap: 5,000/10,000
    bootstrap: 6,000/10,000
    bootstrap: 7,000/10,000
    bootstrap: 8,000/10,000
    bootstrap: 9,000/10,000
    bootstr

In [3]:
m = pd.read_csv("results/external_secondary/metrics.csv")
m = m[m.model.isin(["gene_qn", "all_genes"])]
m["value"] = m.apply(lambda r: f"{r.estimate:.3f} ({r.ci_low:.2f}-{r.ci_high:.2f})", axis=1)
print(m.pivot_table(index=["cohort", "model"], columns="metric", values="value", aggfunc="first").to_string())

metric                          auroc  balanced_accuracy                 f1        sensitivity        specificity
cohort   model                                                                                                   
GSE32280 all_genes  0.641 (0.34-0.91)  0.500 (0.50-0.50)  0.000 (0.00-0.00)  0.000 (0.00-0.00)  1.000 (1.00-1.00)
         gene_qn    0.422 (0.16-0.73)  0.500 (0.50-0.50)  0.000 (0.00-0.00)  0.000 (0.00-0.00)  1.000 (1.00-1.00)
GSE39653 all_genes  0.462 (0.29-0.64)  0.542 (0.40-0.68)  0.571 (0.43-0.70)  0.667 (0.48-0.86)  0.417 (0.21-0.62)
         gene_qn    0.512 (0.34-0.68)  0.500 (0.50-0.50)  0.000 (0.00-0.00)  0.000 (0.00-0.00)  1.000 (1.00-1.00)
